# H₂O Casida TDDFT — Approach 2: Stepwise `CasidaKS_MPI`

This notebook runs the **same physics** as Approach 1, but exposes each Casida stage explicitly:

1. QEpy ground-state SCF
2. Build `CasidaKS_MPI` from the KS density and XC functional
3. Prepare normalized transition orbitals and slice the active space
4. `build_matrices` → `solve` → `oscillator_strengths`
5. Plot stick + broadened spectrum

For the compact one-call workflow, see **`tddft/h2o_tddft_approach1_highlevel.ipynb`**.

## Prerequisites

Same as Approach 1: `qepy`, `dftpy`, `casidapy`, and ONCV UPFs in `tutorials/` (`O_ONCV_PBE-1.2.upf`, `H_ONCV_PBE-1.2.upf` via `paths.py`).


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
TUTORIALS = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "setup_env.sh").is_file() and (_cand / "paths.py").is_file():
        TUTORIALS = _cand
        break
if TUTORIALS is None:
    TUTORIALS = Path("/projectsn/mp1009_1/am4655/casidapy/tutorials")
if str(TUTORIALS) not in sys.path:
    sys.path.insert(0, str(TUTORIALS))
from paths import O_UPF, H_UPF  # noqa: E402
HERE = TUTORIALS
NB_DIR = _cwd

from qepy.driver import Driver

from casidapy.casida_engine import CasidaKS_MPI
from casidapy.utils.casida_utils import normalize_wavefunctions
from casidapy.adapter.qepy import extract_casida_inputs_from_qepy_driver, slice_active_space

try:
    from mpi4py import MPI
    comm = MPI.COMM_WORLD
except ImportError:
    comm = None

HA_TO_EV = 27.211386245988


## 1. Ground-state SCF (shared setup)

In [ ]:
qe_options = {
    "&control": {
        "calculation": "'scf'",
        "pseudo_dir": f"'{HERE}/'",
    },
    "&electrons": {
        "electron_maxstep": 100,
        "conv_thr": 1e-6,
        "mixing_beta": 0.7,
    },
    "&system": {
        "ibrav": 0,
        "nat": 3,
        "ntyp": 2,
        "ecutwfc": 40,
        "ecutrho": 300.0,
        "nosym": True,
        "nbnd": 60,
    },
    "atomic_positions": [
        "O 2.0000000000 2.0000000000 2.1192620000",
        "H 2.0000000000 2.7632390000 1.5229530000",
        "H 2.0000000000 1.2367610000 1.5229530000",
    ],
    "atomic_species": [
        "O 15.999 O_ONCV_PBE-1.2.upf",
        "H 1.008 H_ONCV_PBE-1.2.upf",
    ],
    "k_points automatic": ["1 1 1 0 0 0"],
    "cell_parameters angstrom": [
        "12.0  0.0  0.0",
        "0.0  12.0  0.0",
        "0.0  0.0  12.0",
    ],
}

driver = Driver(qe_options=qe_options, logfile=True)
driver.qe_options["&system"]["nbnd"] = int(len(driver.get_occupation_numbers()) + 2)
scf_energy = driver.scf()
print(f"SCF energy (Ry): {scf_energy:.6f}")

In [ ]:
atoms = driver.get_ase_atoms()
atoms.pbc = True

grid = driver.get_dftpy_grid()
atoms.density = driver.get_density()
atoms.grid = grid
atoms.ions = driver.get_dftpy_ions()
ions = driver.get_dftpy_ions()

# Reuse the adapter only to package wavefunctions / eigenvalues consistently
casida_inputs, _ = extract_casida_inputs_from_qepy_driver(driver, atoms, grid)

## 2. Construct the Casida solver object

Convert the QE density array to a `DirectField` and attach the XC kernel used in the Casida matrix elements.

In [ ]:
rho_ks = driver.data2field(driver.get_density())
xc_func = XC(xc="PBE")

casida = CasidaKS_MPI(rho_ks, xc_func, comm=comm)
print(f"Electrons in cell: {casida.N:.4f}")

## 3. Active space: occupied / unoccupied bands

Count **bands** with non-zero occupation (not total electron count). For H₂O there are four occupied KS bands (occupation 2 each).

In [ ]:
occs = casida_inputs.occs
n_occ = int(np.sum(occs > 0.5))
n_unocc = min(int(np.sum(occs <= 0.5)), 30)  # cap for a faster tutorial
n_states = 25

print(f"Active space: {n_occ} occupied × {n_unocc} unoccupied = {n_occ * n_unocc} transitions")

psi_list = [
    DirectField(grid=casida_inputs.grid, data=casida_inputs.psi[i])
    for i in range(len(casida_inputs.psi))
]
psi_list = [psi / np.sqrt(ions.cell.volume) for psi in psi_list]
psi_list = normalize_wavefunctions(psi_list, casida_inputs.grid)

eigs = np.asarray(casida_inputs.eigs, dtype=float)
occ_e, unocc_e, psi_occ, psi_unocc = slice_active_space(
    eigs,
    psi_list,
    n_occ,
    n_unocc,
    n_total_occ=n_occ,
)
casida.set_active_orbitals(occ_e, unocc_e, psi_occ, psi_unocc)

## 4. Build Casida matrices and solve

- `tda=False` → full Casida (RPA) chain
- `tda=True` would use the Tamm–Dancoff approximation instead

In [ ]:
casida.build_matrices(tda=False)
omega, Z = casida.solve(k=n_states)
f = casida.oscillator_strengths(k=n_states)

omega_ev = omega * HA_TO_EV
for i in range(min(5, len(omega_ev))):
    print(f"  State {i+1:2d}: {omega_ev[i]:8.4f} eV   f = {f[i]:.6f}")

## 5. Plot stick and broadened spectra

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax1.stem(omega_ev, f, basefmt=" ", markerfmt="ro", linefmt="r-")
ax1.set_ylabel("Oscillator strength")
ax1.set_title("Casida stick spectrum (Approach 2)")
ax1.grid(alpha=0.3)

sigma = 0.15
emin = max(0.0, omega_ev.min() - 2 * sigma)
emax = omega_ev.max() + 2 * sigma
grid_ev = np.linspace(emin, emax, 2000)
broadened = np.zeros_like(grid_ev)
for e, osc in zip(omega_ev, f):
    broadened += osc * np.exp(-0.5 * ((grid_ev - e) / sigma) ** 2)

ax2.plot(grid_ev, broadened, "b-", lw=1.8)
ax2.fill_between(grid_ev, broadened, alpha=0.25)
ax2.set_xlabel("Energy (eV)")
ax2.set_ylabel("Absorption (arb. units)")
ax2.set_title(f"Gaussian broadened (σ = {sigma} eV)")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary: when to use Approach 2

| Stage | Method |
|-------|--------|
| Solver object | `CasidaKS_MPI(rho, xc, ...)` |
| Orbitals | manual `DirectField` list + `normalize_wavefunctions` |
| Active space | `slice_active_space` + `set_active_orbitals` |
| Linear algebra | `build_matrices` / `setup_matrix_free` + `solve` |
| Spectroscopy | `oscillator_strengths` |

Choose this path when you need fine control (TDA vs RPA, matrix-free solvers, USPP, triplet, coupling to eDFTpy transition densities, etc.) without going through `run_casida_in_memory`.